# Part 1: 에이전트형 검색으로 지식 베이스 구축하기

방금 Zava 출입 배지와 노트북을 받았습니다. 첫 번째 임무는 회사 전체가 HR 및 복리후생 질문에 답하는 데 사용할 AI 기반 지식 시스템을 구축하는 것입니다.

**🎯 미션**
- 클라우드 아키텍처 PDF를 **파일 지식 소스(File Knowledge Source)** 에 직접 업로드하고 질의하기
- 미리 구축된 두 개의 검색 인덱스(**HR 문서**와 **건강 복리후생 문서**)를 **색인된 지식 소스(Indexed Knowledge Source)** 로 연결하기
- 각 하위 질문을 적절한 소스로 자동 라우팅하는 멀티 소스 **지식 베이스(Knowledge Base)** 구축하기


## 빌딩 블록

Foundry IQ(Azure AI Search)에서 **지식 베이스**는 채팅 완성 모델을 하나 이상의 지식 소스에 연결하는 최상위 리소스입니다. 어떤 데이터 소스를 질의할지, 추론에 어떤 모델을 사용할지, 질의 실행을 어떻게 최적화할지를 정의합니다.

이 노트북에서는 두 가지 유형의 지식 소스를 살펴봅니다.

1. **파일 지식 소스(File Knowledge Source)**: 원시 파일을 직접 업로드합니다. 서비스가 청킹, 임베딩, 색인을 자동으로 처리합니다.
2. **색인된 지식 소스(Indexed Knowledge Source)**: 스키마와 처리 과정을 완전히 제어할 수 있는 프로덕션 워크로드를 위해 미리 구축된 검색 인덱스에 연결합니다.

## 미리 구성된 환경

이 워크샵은 다음 Azure 리소스를 사용하며, 이미 설정되어 있습니다.

* Azure AI Search
  * `hrdocs` 인덱스: HR 정책, 핸드북, 회사 정보
  * `healthdocs` 인덱스: 건강 복리후생, 보험, 보장
  * 의미 체계 순위(semantic ranking) 활성화, 사전 계산된 임베딩
* Microsoft Foundry Models의 Azure OpenAI
  * `gpt-5.4`: 추론 및 합성을 위한 채팅 완성
  * `text-embedding-3-large`: 벡터 검색을 위한 임베딩 모델
* Microsoft Fabric

## 1단계: 환경 변수 설정

Azure 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 필요한 모든 것이 들어 있습니다. Azure AI Search 엔드포인트, Azure OpenAI 자격 증명, 모델 구성입니다.

**ℹ️ 참고**
- 아래 셀을 처음 실행하면 커널을 선택하라는 메시지가 표시됩니다. **Python Environments**를 선택하고 **.venv** 환경을 고르세요.
- "이 앱이 퍼블릭 및 프라이빗 네트워크에 액세스하도록 허용하시겠습니까?"라는 메시지가 표시될 수도 있습니다. **허용**을 선택하세요.

**⚠️ 문제 해결:**
코드 셀이 멈춰서 계속 도는 경우 다음 해결 단계를 시도하세요.
1. 상단의 노트북 도구 모음에서 **Restart**를 선택합니다. 그래도 안 되면:
2. VS Code 명령 팔레트에서 **Reload window**를 선택합니다. 그래도 안 되면:
3. VS Code를 완전히 닫고 다시 시작합니다.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


---

## A: 파일 지식 소스(File Knowledge Source)

**파일 지식 소스(File Knowledge Source)** 를 사용하면 원시 파일을 직접 업로드할 수 있습니다. 서비스가 청킹, 임베딩, 색인을 자동으로 처리합니다. 문서를 지식 베이스에 넣는 가장 빠른 방법입니다.

## 2단계: 파일 지식 소스 만들기

자동 수집을 위해 임베딩 모델과 함께 파일 지식 소스를 구성합니다. `ingestion_parameters`는 다음을 지정합니다.
- **임베딩 모델**: 업로드된 콘텐츠에 대한 벡터 임베딩을 생성합니다
- **콘텐츠 추출 모드**: 문서에서 텍스트를 추출하는 방식입니다

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Uploaded HR and company documents for Zava",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## 3단계: 지식 소스에 파일 업로드

파일 지식 소스에 문서를 직접 업로드합니다. 서비스가 자동으로 텍스트를 추출하고, 검색 가능한 세그먼트로 청킹하고, 벡터 임베딩을 생성합니다.

⏱️ 샘플 파일을 수집하는 데 약 40초가 걸립니다.

In [ ]:
from pathlib import Path

data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## 4단계: 파일 소스를 위한 지식 베이스 만들기

지식 베이스는 다음을 결합하는 오케스트레이션 계층입니다.

1. 데이터 소스(파일 지식 소스)
2. 질의를 이해하고 답변을 생성하기 위한 LLM(Azure OpenAI)
3. 질의를 처리하고 응답 형식을 지정하는 방법에 대한 구성

이 지식 베이스는 `ANSWER_SYNTHESIS` 출력 모드를 사용하므로, 연결된 LLM이 검색된 청크로부터 완전한 자연어 답변을 생성합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "file-docs-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## 5단계: 파일 지식 베이스에 질의

업로드한 문서에 대해 질문합니다. 지식 베이스가 파일 콘텐츠 전체를 검색하고 인용이 포함된 자연어 답변을 합성합니다.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "As a new Zava engineer, what cloud architecture principles should I know about? What data sensitivity levels does Zava use?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

### 활동 로그 검토

활동 로그는 지식 베이스가 수행한 단계의 순서를 보여줍니다. 각 활동에는 다음이 포함됩니다.

* `type`: 활동 유형입니다. 이 로그에서는 첫 번째 단계인 "modelQueryPlanning", 여러 개의 "file" 단계(파일 지식 소스로 보낸 각 질의마다 하나씩), 답변 합성 단계인 "modelAnswerSynthesis", 의미 체계 재순위 단계인 "agenticReasoning"을 볼 수 있습니다.
* `elapsedMS`: 활동의 소요 시간입니다.
* `id`: 활동 ID로, 각 참조를 그것을 생성한 활동과 연결하는 데 유용합니다. 다음으로 참조를 검토합니다.

활동 항목에는 유형별로 고유한 다른 메타데이터도 포함되어 있어, 모델 기반 단계에서 사용된 토큰 수를 확인하고 검색 단계에서 소스로 보낸 질의를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### 참조 검토

다음 셀은 지식 베이스가 반환한 원시 `references` 배열을 출력합니다.

참조는 어떤 소스가 검색되었는지와 각 참조의 소스 메타데이터를 보여줍니다. 모든 결과에는 다음이 포함됩니다.

* `type`: 이 지식 베이스에는 파일 소스만 포함되어 있으므로 항상 "file"입니다.
* `id`: 참조의 숫자 식별자입니다. 답변에서 `[ref:1]` 같은 인용 마커로 나타나는 값입니다.
* `activitySource`: 이 참조를 생성한 활동 로그의 검색 활동 id입니다. 어떤 검색 단계가 이 문서를 반환했는지 추적하는 데 사용할 수 있습니다.
* `sourceData`: 자동 생성된 검색 인덱스에서 검색된 청크의 필드입니다. 파일 소스 인덱스의 경우 항상 `uid`, `snippet`, `metadata_storage_path`입니다.
* `rerankerScore`: 재순위 모델이 매긴 0~4 사이의 점수로, 4가 가장 좋습니다.
* `docName`: 업로드된 파일에 대해 자동 생성된 파일 이름입니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: 색인된 지식 소스(Indexed Knowledge Sources)

**색인된 지식 소스(Indexed Knowledge Sources)** 는 미리 구축된 검색 인덱스에 연결하여 스키마, 청킹, 임베딩을 완전히 제어할 수 있게 해줍니다. 대규모 데이터셋을 다루는 프로덕션 워크로드에 이상적입니다.

이 섹션에서는 수집된 문서 청크가 채워진 두 개의 검색 인덱스를 이미 미리 채워 두었습니다. 일반적으로는 인덱서 파이프라인을 설정하여 Azure Blob Storage 같은 소스에서 검색 인덱스로 문서를 수집합니다.

## 6단계: 검색 인덱스 확인

아래 셀을 실행하여 각 인덱스에 예상한 수의 문서 청크가 있는지 확인합니다.

* `hrdocs`: HR 정책에 대한 50개의 문서 청크
* `healthdocs`: 건강 복리후생 및 보험에 대한 334개의 문서 청크

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## 7단계: 두 개의 색인된 지식 소스 만들기

각 인덱스를 가리키는 두 개의 지식 소스를 만듭니다.

* `healthdocs-knowledge-source`: `healthdocs` 인덱스를 가리킴
* `hrdocs-knowledge-source`: `hrdocs` 인덱스를 가리킴

두 소스 모두 `snippet` 필드를 검색 가능 필드로 설정하고, `blob_path`를 소스 데이터 필드로 포함하여 해당 경로를 인용 링크에 사용할 수 있도록 합니다.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## 8단계: 멀티 소스 지식 베이스 만들기

두 개의 검색 인덱스 지식 소스를 모두 포함하는 지식 베이스를 만듭니다. 이전 지식 베이스와 마찬가지로 이 지식 베이스도 `output_mode=ANSWER_SYNTHESIS`를 사용하므로, 연결된 LLM을 사용해 근거가 포함된 인용으로 원래 질의에 답합니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## 9단계: 멀티 소스 지식 베이스에 질의

두 지식 소스에 걸쳐 있는 두 부분으로 된 온보딩 질문을 해봅시다.

* _"Zava 시니어 직원은 휴가가 몇 주인가요?"_ → HR 관련, `hrdocs`에서 와야 함
* _"정신 건강 서비스에 가장 좋은 보장을 제공하는 건강 플랜은 무엇인가요?"_ → 복리후생 관련, `healthdocs`에서 와야 함

지식 베이스는 에이전트형 검색을 사용해 다음을 수행합니다.

1. 질의를 분석하고 두 개의 서로 다른 주제를 인식
2. 집중된 하위 질의로 분해
3. 각 하위 질의에 적합한 지식 소스 선택
4. 두 소스에 걸쳐 검색을 동시 실행
5. 결과를 병합하여 반환

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "I just joined Zava as a senior engineer. "
    "According to company policy, how many vacation weeks does a senior employee get? "
    "And among the available health plans, which gives the best coverage for mental health services?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))

### 활동 로그 검토

이 활동 로그에서는 여러 개의 "searchIndex" 단계를 볼 수 있으며, 각 검색 인덱스로 보낸 질의마다 하나씩입니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### 참조 검토

이번에는 참조의 type이 "searchIndex"이며, 이 검색 인덱스가 자동 생성된 파일 소스 인덱스와는 다른 필드 구성을 가지고 있으므로 "sourceData" 필드도 약간 다릅니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 소스 헌트

위의 참조와 활동 로그를 살펴보세요. 다음을 찾을 수 있나요?

1. 어떤 지식 소스가 **휴가** 질문에 답했나요?
2. 어떤 지식 소스가 **정신 건강 플랜** 질문에 답했나요?
3. 에이전트가 **두** 소스를 모두 질의했나요, 아니면 각 하위 질의에 대해 가장 관련성 높은 소스만 질의했나요?

## 보너스: Copilot CLI에서 질의하기

모든 지식 베이스는 MCP 서버 엔드포인트를 노출하며, 이는 VS Code나 GitHub Copilot CLI 같은 MCP 호환 클라이언트에 추가할 수 있습니다.
Copilot CLI에서 지식 베이스에 질의해 보고 싶다면 [Copilot CLI 사이드퀘스트](copilot-cli-sidequest.md)의 설정 단계를 거친 후 아래 명령을 실행하여 Zava 지식 베이스를 MCP 서버로 사용하세요.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add zava-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print("copilot -i \"I just joined Zava as a senior engineer. According to company policy, how many vacation weeks does a senior employee get?\"")

## ✅ 미션 완료

**무엇을 만들었나:**

* ✓ **파일 지식 소스(File Knowledge Source)**: PDF를 직접 업로드할 수 있고 모든 청킹과 임베딩을 자동으로 처리하는 소스.
* ✓ **색인된 지식 소스(Indexed Knowledge Sources)**: 미리 구축된 HR 및 건강 검색 인덱스를 위한 소스로, 인덱스 스키마와 처리 과정을 완전히 제어할 수 있음.
* ✓ **멀티 소스 지식 베이스(Multi-source Knowledge Base)**: 각 하위 질문을 적절한 소스로 자동 라우팅하고 인용 기반 답변을 제공하는 단일 지식 베이스.

➡️ [Part 2: 웹 결과로 근거 추가하기](part2-search-mcp-kb.ipynb)로 계속 진행하세요.